In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Colab/tested.csv')

print(df.head())

   PassengerId  Survived  Pclass  \
0          892         0       3   
1          893         1       3   
2          894         0       2   
3          895         0       3   
4          896         1       3   

                                           Name     Sex   Age  SibSp  Parch  \
0                              Kelly, Mr. James    male  34.5      0      0   
1              Wilkes, Mrs. James (Ellen Needs)  female  47.0      1      0   
2                     Myles, Mr. Thomas Francis    male  62.0      0      0   
3                              Wirz, Mr. Albert    male  27.0      0      0   
4  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female  22.0      1      1   

    Ticket     Fare Cabin Embarked  
0   330911   7.8292   NaN        Q  
1   363272   7.0000   NaN        S  
2   240276   9.6875   NaN        Q  
3   315154   8.6625   NaN        S  
4  3101298  12.2875   NaN        S  


In [3]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)
# Pclass:
# 1 = First Class
# 2 = Second Class
# 3 = Third Class

y = df["Pclass"]
features = [
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = df[features].copy()
categorical_features = [
    "Sex",
    "Embarked"
]
numerical_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]
for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)
multinomial_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)
multinomial_model.fit(X_train, y_train)
y_pred = multinomial_model.predict(X_test)

y_probability = multinomial_model.predict_proba(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\n============================================")
print("MULTINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "First Class",
            "Second Class",
            "Third Class"
        ]
    )
)



MULTINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.75

Confusion Matrix:
[[17  3  1]
 [ 2  3 14]
 [ 0  1 43]]

Classification Report:
              precision    recall  f1-score   support

 First Class       0.89      0.81      0.85        21
Second Class       0.43      0.16      0.23        19
 Third Class       0.74      0.98      0.84        44

    accuracy                           0.75        84
   macro avg       0.69      0.65      0.64        84
weighted avg       0.71      0.75      0.71        84



/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
